# GuindiMa AI — nouvelle voix wolof : CONCREE/Adia_TTS (sur Colab)

Ce notebook **régénère tout `data/tts_cache/`** avec la voix **Adia_TTS**
(`CONCREE/Adia_TTS`), qui remplace la voix SpeechT5 actuelle.

- Modèle : `parler-tts-mini-multilingual-v1.1` affiné sur 40 h de wolof.
- Licence : **Apache 2.0** (usage commercial autorisé, mentionner CONCREE).
- Sortie : **24 kHz**, 16-bit PCM.
- **GPU obligatoire** (Runtime > Modifier le type d'exécution > GPU, T4 suffit).

Les phrases à dire sont lues depuis `scripts/phrases_voix.json` (généré depuis
`src/tts/pregenerer.py` : 106 lignes du jeu de données + 2 phrases de service,
soit **109 fichiers**). Les nombres sont déjà écrits en mots wolof, car le
modèle lit mal les chiffres. À l'oral on dit **« numero »** (plus clair que
« ligne ») ; l'affichage des cartes garde « Ligne 46 ».

À la fin, on télécharge `tts_cache_adia.zip` : `scripts/installer_voix_adia.py`
s'occupera de remplacer le cache en local (avec sauvegarde de l'ancienne voix).

In [ ]:
# Un GPU doit être attribué, sinon : Runtime > Change runtime type > GPU.
!nvidia-smi

## Installation

1. `parler-tts` (paquet officiel Hugging Face).
2. Colab embarque une version ancienne de `protobuf` (via TensorFlow) ;
   `parler-tts`/`transformers` exigent `protobuf>=5.29`, sinon on tombe sur
   `cannot import name 'runtime_version' from 'google.protobuf'`.

**Après la cellule suivante : redémarre la session**
(`Runtime > Restart session`), puis continue directement à la cellule de
chargement du modèle (ne réinstalle pas).

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q "protobuf>=5.29"
print("Installé. Redémarre maintenant la session (Runtime > Restart session).")

In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODELE = "CONCREE/Adia_TTS"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
assert device == "cuda:0", "Colab doit tourner sur GPU (Runtime > Change runtime type > GPU)"

model = ParlerTTSForConditionalGeneration.from_pretrained(MODELE).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODELE)
print("Chargé :", MODELE, "| fréquence :", model.config.sampling_rate)

In [ ]:
# Récupérer scripts/phrases_voix.json (dépose-le depuis le dépôt local).
import json
import os
from google.colab import files

if not os.path.exists("phrases_voix.json"):
    print("Choisis scripts/phrases_voix.json…")
    envoi = files.upload()
    nom = next(iter(envoi))
    os.rename(nom, "phrases_voix.json")

with open("phrases_voix.json", encoding="utf-8") as f:
    plan = json.load(f)

GRAINE = plan.get("graine", 1234)
DESCRIPTION = plan.get("description", "A warm and natural voice, with a conversational flow")
PHRASES = plan["phrases"]
print(len(PHRASES), "phrases |", DESCRIPTION)

In [ ]:
# Générer les 109 WAV : coupe des silences au début/fin, puis égalisation du volume.
import os
import numpy as np
import soundfile as sf

SORTIE = "tts_cache_adia"
os.makedirs(SORTIE, exist_ok=True)
SR = model.config.sampling_rate


def nettoyer(audio, sr, marge=0.05, cible=0.97):
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    seuil = max(1e-3, 0.02 * float(np.max(np.abs(audio))))
    forts = np.where(np.abs(audio) > seuil)[0]
    if forts.size:
        debut = max(0, forts[0] - int(sr * marge))
        fin = min(len(audio), forts[-1] + int(sr * marge))
        audio = audio[debut:fin]
    crete = float(np.max(np.abs(audio))) or 1.0
    return (audio * (cible / crete)).astype(np.float32)


for i, item in enumerate(PHRASES):
    chemin = os.path.join(SORTIE, item["fichier"])
    if os.path.exists(chemin):
        continue
    torch.manual_seed(GRAINE)  # même voix d'un bout à l'autre
    entrees = tokenizer(DESCRIPTION, return_tensors="pt").input_ids.to(device)
    invite = tokenizer(item["phrase"], return_tensors="pt").input_ids.to(device)
    with torch.inference_mode():
        audio = model.generate(input_ids=entrees, prompt_input_ids=invite)
    sf.write(chemin, nettoyer(audio.float().cpu().numpy(), SR), SR, subtype="PCM_16")
    if (i + 1) % 10 == 0 or i == 0:
        print(f"{i + 1}/{len(PHRASES)}  {item['fichier']}")

produits = sorted(os.listdir(SORTIE))
print(f"\n{len(produits)} / {len(PHRASES)} fichiers dans {SORTIE}/")
manquants = [p["fichier"] for p in PHRASES if p["fichier"] not in produits]
print("manquants :", manquants or "aucun")

In [ ]:
# Écoute rapide d'un exemple avant de tout empaqueter.
from IPython.display import Audio, display

exemple = os.path.join(SORTIE, PHRASES[0]["fichier"])
print(PHRASES[0]["phrase"])
display(Audio(exemple))

In [ ]:
# Empaqueter le dossier et le télécharger.
import shutil
from google.colab import files

shutil.make_archive("tts_cache_adia", "zip", SORTIE)
files.download("tts_cache_adia.zip")